# RAG Evaluationwith RAGAS

**Module 8 · Lesson 8.11**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- The 4 Metrics From Scratch - Understand Before You Automate
- RAGAS Library - Automated Evaluation on the Current API
- Synthetic Test Data - Generate Your Golden Set
- The Improvement Loop - A/B Test Every Change
- DeepEval - Evaluation as pytest
- Retrieval Metrics - NDCG, MRR, Precision@k, Recall@k

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## “It Seems to Work” Isn’t Engineering

> 💡 **Info**
>
> Your RAG assistant answers “What’s the refund policy?” with: “Refunds are processed within 14 business days. You need form RF-201. **The company was founded in 2020.**” The first two facts are in your documents. The third is invented - the model hallucinated a founding date that appears in no retrieved chunk. It *reads* perfectly. A demo audience nods. And nothing in your pipeline catches it, because you have no metric that asks “is every claim actually supported by the context?” **That single unanswerable question is why this lesson exists.** RAG evaluation gives you numbers - faithfulness, relevance, precision, recall - that flag the hallucination, tell you whether a new chunking strategy helped or hurt, and turn “it seems to work” into something you can measure, compare, and gate a deployment on.

### 🎯 What you will be able to do after this lesson

- **Build** the four core RAG metrics from scratch (faithfulness, answer relevance, context precision, context recall) as LLM-judge calls, so you know exactly what a framework measures.

- **Operate** the current RAGAS library and DeepEval: automated evaluation, synthetic test-set generation, and pytest CI gates.

- **Compare** components in isolation - separate retrieval metrics (NDCG/MRR/Recall@k) from generation faithfulness - and A/B test every change.

- **Evaluate** trustworthiness: LLM-as-judge biases, human-in-the-loop calibration (Cohen’s Kappa), production observability, and India multilingual/legal eval.

> 📦 **Info**
>
> 🧭 Before you start
> You need everything you built in 8.1–8.10 (any RAG pipeline to evaluate), a Google AI Studio key in `GOOGLE_API_KEY`, and `pip install google-genai` for the by-hand steps (plus `ragas langchain-google-genai` and `deepeval` for the framework steps). Every block uses the current **google-genai** SDK and the current **RAGAS v0.2+** API: pre-2025 tutorials import the retired `google.generativeai` and RAGAS’s old `Dataset.from_dict` API, and error on today’s stack.

## 60-Second Warm-Up: What You Already Know

Three recalls, all load-bearing today. Click a card to check yourself.

> 🏥 **Analogy**
>
> **An unevaluated RAG system is a hospital that never checks patient outcomes.** Doctors prescribe, patients leave, and nobody tracks whether anyone actually got better. “The doctor seemed confident” is not a quality metric. Real hospitals track recovery rates, infection rates, readmission rates - specific, measurable, actionable numbers. RAG evaluation is those numbers: did retrieval find the right chunks (precision/recall), did the answer stay faithful to them (faithfulness), did it address the question (relevance)?
> **Where the analogy breaks down:** a hospital’s outcomes are ground truth - the patient recovered or didn’t. Your RAG “metrics” are mostly an LLM judging another LLM, which has biases and gets languages wrong, so the numbers are estimates, not verdicts. That is why the later steps add rank-aware retrieval metrics, human calibration (Cohen’s Kappa), and negative controls - you evaluate the evaluator, too.

> 💡 **Info**
>
> ⚠️ Misconception: “a high faithfulness score means the RAG is good”
> The single most common evaluation mistake. Faithfulness only measures whether the answer is grounded in the RETRIEVED context - it says nothing about whether that context is correct or complete. You can score 0.92 faithfulness and give terrible answers, because the model is faithfully summarizing the WRONG documents. High faithfulness + bad answers is the classic signature of a RETRIEVAL failure, not a generation one. That is why you never look at one number: you evaluate retrieval (precision, recall, NDCG) and generation (faithfulness, relevance) SEPARATELY, so you know which half to fix. A single score hides the bug; component isolation finds it.

> 💡 **Info**
>
> ⛔ Anti-pattern: “eyeball a few answers and ship”
> The tempting **wrong way** to “evaluate” RAG is to read five answers, decide they look good, and deploy. It **fails because** a hand-picked demo hides the tail: the hallucinated sub-clause, the query your chunker splits badly, the language your judge is weak on. Worse, “it seems better” **breaks when** you compare two versions - you cannot tell a real gain from luck without a test set and a metric. Don’t do this: build a test set (step 3), measure the four metrics on it, gate changes on the numbers, and treat sub-noise deltas as ties. A vibe is not a measurement.

## Build One: The Metrics, By Hand

Steps 1–4 build RAG evaluation from scratch: the four core metrics as LLM-judge calls, then the RAGAS library, synthetic test data, and an A/B improvement loop. Building each metric once is what makes the frameworks legible - RAGAS and DeepEval are industrial versions of these exact judge calls.

---

## 🎯 Concept 1: The 4 Metrics From Scratch - Understand Before You Automate

### The 4 Metrics From Scratch - Understand Before You Automate

Each metric is one LLM-judge call returning structured JSON. Build them so you know what RAGAS measures.

**Why this is step 1:** before you trust a framework’s number, you should know how it was computed. Each of the four metrics is a small LLM-judge call with a specific prompt and a 0–1 score. Faithfulness decomposes the answer into claims and checks each; relevance rates how well the answer addresses the question; precision/recall judge the retrieved chunks. Write them once and RAGAS stops being magic.

> 🧷 **Analogy**
>
> **The judge is a fact-checker with a highlighter.** Give a fact-checker your answer and your sources, and they underline each claim, then flag the ones the sources don’t support. Faithfulness is exactly that: break the answer into claims, check each against the context, score = supported / total. “Founded in 2020” gets no underline - it’s in no source - so the score drops.
> **Where the analogy breaks down:** a human fact-checker has real-world knowledge; the LLM judge only compares text to text, so if the retrieved context itself is wrong, the judge happily marks a wrong-but-grounded claim as “supported.” Faithfulness measures grounding, not truth - which is the whole reason you also measure retrieval quality separately in step 6.

Faithfulness breaks the answer into claims and scores supported / total. What does the “founded in 2020” claim do to the score?

- Each metric method builds a prompt, calls a temperature-0 judge, and parses a JSON score.

- **faithfulness** decomposes into claims; **answer_relevance** rates the answer vs the question; **context_precision** judges each retrieved chunk; **context_recall** checks the ground-truth facts against the chunks.

- **evaluate** runs them all and averages - the same shape RAGAS returns.

**📝 `01_metrics_scratch.py`** - *google-genai*

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install google-genai langchain-core langchain-google-genai deepeval ragas python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


#### 💡 What just happened

⚡ What just happened?Four separate LLM-judge calls scored one Q&A. Faithfulness caught “founded in 2020” as an unsupported claim (0.67); context precision flagged the Docker chunk as retrieval noise (0.50); recall confirmed the real facts were present (1.00). Each number points at a different fix. The tradeoff of the judge approach: it’s automated and scalable but non-deterministic (temperature 0 helps, not perfectly) and only measures text-vs-text grounding, not truth - which is why step 6 adds rank-aware retrieval metrics and step 9 adds human calibration.

- Toggle each claim in the answer between “supported” and “hallucinated” and watch faithfulness recompute.
- Add or remove a noisy retrieved chunk and see context precision move.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: RAGAS Library - Automated Evaluation on the Current API

### RAGAS Library - Automated Evaluation on the Current API

Same metrics, batched. The current v0.2+ API uses EvaluationDataset and metric classes.

**Why this is step 2:** now that you know what the metrics compute, let the library do it at scale. RAGAS handles batching, retries, and reporting - but its API changed. The old `Dataset.from_dict({"question","answer","contexts","ground_truth"})` is deprecated (see the **RAGAS** repo at [github.com/explodinggradients/ragas](https://github.com/explodinggradients/ragas)); the current API uses `EvaluationDataset.from_list` with renamed fields (`user_input`, `response`, `retrieved_contexts`, `reference`) and metric CLASSES you instantiate.

> 🏭 **Analogy**
>
> **Your hand-built metrics were a home kitchen; RAGAS is the commercial line.** Same recipe, but now it plates a hundred orders while you watch. You hand it a dataset of question/answer/context/reference rows and it returns a per-row scorecard for every metric - the thing production teams re-run after every pipeline change.
> **Where the analogy breaks down:** a commercial kitchen’s recipes are fixed; RAGAS’s API is not - the field names and metric classes were renamed between v0.1 and v0.2, so a copy-pasted 2024 snippet errors today. Pin your RAGAS version and use the current `EvaluationDataset` shape, or you’ll debug import errors instead of your RAG.

A 2024 tutorial builds the RAGAS dataset with columns `question / answer / contexts / ground_truth`. Will it run on current RAGAS?

- Each sample uses the RENAMED fields: `user_input`, `retrieved_contexts`, `response`, `reference`.

- `EvaluationDataset.from_list` replaces `Dataset.from_dict`; metrics are CLASSES you instantiate (`Faithfulness()`).

- `LangchainLLMWrapper` wraps a Gemini chat model as the judge; `evaluate` returns aggregate + per-sample scores.

**📝 `02_ragas_library.py RAGAS`** - *v0.2+*

In [ ]:
# RAGAS LIBRARY - the CURRENT API (EvaluationDataset + metric classes)
# pip install ragas langchain-google-genai
from ragas import EvaluationDataset, evaluate
from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithReference, LLMContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

judge = LangchainLLMWrapper(ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0))
emb = LangchainEmbeddingsWrapper(GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001"))

# each row uses the CURRENT field names (question->user_input, answer->response,
# contexts->retrieved_contexts, ground_truth->reference)
samples = [
    {"user_input": "What is the refund policy?",
     "retrieved_contexts": ["Refunds processed within 14 business days. Submit form RF-201 with order ID."],
     "response": "Refunds are processed within 14 business days via form RF-201.",
     "reference": "Refunds take 14 business days. Form RF-201 required with order ID."},
    {"user_input": "What GPU is required?",
     "retrieved_contexts": ["GPU acceleration requires CUDA 12.1+. Minimum 8 GB VRAM."],
     "response": "You need CUDA 12.1+ and 8 GB VRAM minimum.",
     "reference": "CUDA 12.1 or later. 8 GB VRAM minimum."},
]
dataset = EvaluationDataset.from_list(samples)

result = evaluate(
    dataset=dataset,
    metrics=[Faithfulness(), ResponseRelevancy(),
             LLMContextPrecisionWithReference(), LLMContextRecall()],
    llm=judge, embeddings=emb,
)
print(result)                 # aggregate scores across the dataset
print(result.to_pandas())     # per-sample breakdown as a DataFrame

# Output:
# {'faithfulness': 1.00, 'answer_relevancy': 0.94, 'llm_context_precision_with_reference': 1.00, 'context_recall': 1.00}
# (plus a per-row DataFrame: swap a chunking strategy, re-run, compare the columns)

#### 💡 What just happened

⚡ What just happened?The official RAGAS library scored the same metrics as step 1, batched over a dataset, with Gemini as the judge - on the CURRENT API (`EvaluationDataset.from_list`, metric classes, the renamed `user_input`/`response`/`retrieved_contexts`/`reference` fields). This is the production loop: change a chunker or prompt, re-run `evaluate`, compare the columns; if faithfulness drops, the change added hallucination. The tradeoff vs your hand-built version: the library gives you batching, retries, and a DataFrame - but you had to build it once to know that “answer_relevancy” is the same claim-vs-question judge you wrote in step 1. The RAGAS class names map straight back to those four: `Faithfulness`=faithfulness, `ResponseRelevancy`=answer relevance (result key `answer_relevancy`), `LLMContextPrecisionWithReference`=context precision, `LLMContextRecall`=context recall - “WithReference” just means it uses your reference answer to judge precision.

- Slide top-k over a labelled corpus and watch precision fall while recall rises.
- See why “high precision, low recall” means the retriever is too selective.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Synthetic Test Data - Generate Your Golden Set

### Synthetic Test Data - Generate Your Golden Set

Good evaluation needs good test data. RAGAS TestsetGenerator builds it from your documents.

**Why this is step 3:** metrics are useless without questions to run them on, and hand-writing 200 test questions takes weeks. RAGAS’s `TestsetGenerator` builds a knowledge graph from your documents and synthesizes diverse Q&A pairs - single-hop factual and multi-hop reasoning - each with a reference answer and supporting context. You then keep a small human-curated golden set for calibration and lean on synthetic data for coverage.

> 📝 **Analogy**
>
> **Synthetic test generation is an exam-writer who read the textbook.** Hand it your documents and it drafts a question bank - easy recall questions and harder ones that require connecting two sections - complete with an answer key. You save weeks of writing questions yourself; you just review and cull.
> **Where the analogy breaks down:** an exam-writer knows what real students struggle with; a generator only knows your documents, so its questions may not match how real users actually ask (typos, ambiguity, out-of-scope). That’s why the production pattern is HYBRID: a small golden set of real, expert-verified questions for calibration, synthetic data for coverage, and real production failures fed back in continuously.

You auto-generate 300 synthetic test questions from your docs. Can you delete your small hand-written golden set?

- `TestsetGenerator(llm=, embedding_model=)` builds a knowledge graph from your documents.

- `generate_with_langchain_docs(docs, testset_size=N)` synthesizes N Q&A pairs across single-hop and multi-hop query types.

- Each row has a `user_input` and a `reference` answer - a ready RAGAS evaluation set.

**📝 `03_testset.py RAGAS`** - *TestsetGenerator*

In [ ]:
# SYNTHETIC TEST DATA - RAGAS TestsetGenerator (knowledge-graph based, v0.2+)
# pip install ragas langchain-google-genai
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

gen_llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(model="gemini-2.5-flash"))
gen_emb = LangchainEmbeddingsWrapper(GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001"))

docs = [Document(page_content=c) for c in [
    "The NetsAI course costs 29,999 rupees/year or 2,999/month. Enterprise from 4,99,000/year.",
    "Refunds are processed within 14 business days. Submit form RF-201 with your order ID.",
    "GPU acceleration requires CUDA 12.1+ and 8 GB VRAM minimum."]]

generator = TestsetGenerator(llm=gen_llm, embedding_model=gen_emb)
testset = generator.generate_with_langchain_docs(docs, testset_size=6)
# default query mix: SingleHopSpecific 0.5, MultiHopAbstract 0.25, MultiHopSpecific 0.25

df = testset.to_pandas()
print(df[["user_input", "reference"]].head())

# Output: (KG-synthesized Q&A, ready to feed step 2's evaluate())
# user_input                                       reference
# What form is needed to request a refund?         Form RF-201, submitted with the order ID.
# What are the GPU requirements for the course?    CUDA 12.1 or later and at least 8 GB VRAM.
# ... 6 rows: a mix of single-hop and multi-hop questions

#### 💡 What just happened

⚡ What just happened?RAGAS built a knowledge graph from three documents and synthesized six Q&A pairs - a ready evaluation set, in seconds instead of days. The v0.2 generator replaced the old simple/reasoning/multi_context “evolution types” with named synthesizers (single-hop specific, multi-hop abstract/specific). The tradeoff: synthetic questions are fast and broad but may not match how real users ask, so treat them as coverage, keep a small expert-verified golden set for calibration, and continuously fold in real production failures - a sample of a few hundred gives you a tight confidence interval.

---

## 🎯 Concept 4: The Improvement Loop - A/B Test Every Change

### The Improvement Loop - A/B Test Every Change

Change the chunker, prompt, or model? Measure the impact with metrics before you deploy.

**Why this is step 4:** metrics only earn their keep when they drive decisions. The improvement loop is: measure a baseline, change one thing, re-evaluate on the SAME test set, and deploy only if the scores went up. This turns every RAG tweak from a hunch into an experiment - and the diagnose step tells you WHICH metric to chase.

> 📊 **Analogy**
>
> **The improvement loop is A/B testing for RAG.** You don’t ship a new checkout button because it “feels better” - you run A vs B on the same traffic and keep the winner. Same here: run RAG v1 and v2 on the same questions, compare all four metrics side by side, and deploy the one with the higher scores - with a diagnosis of what each is weak at.
> **Where the analogy breaks down:** web A/B tests have millions of users and clean conversion signals; your eval set has a few hundred questions and an LLM judge with noise, so a tiny (~1%) metric difference is probably not real. Change ONE variable at a time, use enough test questions, and treat small deltas as ties - otherwise you’ll “improve” on judge noise.

You change the chunker AND the prompt AND the model at once, and faithfulness goes up 0.04 (4 points, where 1 point = 0.01 on the 0-1 scale). What do you know?

- `run_experiment` scores a RAG pipeline over the eval set with the step-1 evaluator.

- `compare` prints all experiments side by side with improvement/regression deltas.

- `diagnose` maps each low metric to a concrete fix - low faithfulness → grounding prompt, low recall → raise top-k.

**📝 `04_experiment.py A/B`** - *Harness*

In [ ]:
# THE IMPROVEMENT LOOP - A/B test RAG changes with the metrics from step 1
class RAGExperiment:
    def __init__(self, eval_set):
        self.eval_set, self.evaluator, self.results = eval_set, RAGEvaluator(), {}

    def run(self, name, rag_fn):
        # run a pipeline on every test question, score it, average each metric over the rows that have it
        rows = [self.evaluator.evaluate(qa["question"], (r := rag_fn(qa["question"]))["answer"],
                                        r["contexts"], qa.get("answer", "")) for qa in self.eval_set]
        self.results[name] = {k: sum(r[k] for r in rows if k in r) / sum(1 for r in rows if k in r) for k in rows[0]}
        return self.results[name]

    def compare(self):
        names = list(self.results)
        for m in self.results[names[0]]:
            vals = [self.results[n][m] for n in names]
            delta = vals[-1] - vals[0]
            arrow = "up" if delta > 0.02 else ("down" if delta < -0.02 else "tie")
            print(f"  {m:20s} " + " ".join(f'{v:.2f}' for v in vals) + f"   {arrow} {delta:+.2f}")
        best = max(names, key=lambda n: self.results[n]["overall"])
        print(f"  winner: {best}")

test_set = [{"question": "What is the refund policy?", "answer": "14 business days, form RF-201."}]
exp = RAGExperiment(test_set)
exp.run("v1_basic", lambda q: {"answer": "Refunds take 14 days.", "contexts": ["Refunds within 14 business days."]})
exp.run("v2_grounded", lambda q: {"answer": "Refunds take 14 business days; submit form RF-201.", "contexts": ["Refunds within 14 business days. Submit RF-201."]})
exp.compare()

# Output:
#   faithfulness         1.00 1.00   tie +0.00
#   answer_relevance     0.80 0.95   up  +0.15   <- v2 answers more completely
#   context_precision    1.00 1.00   tie +0.00
#   context_recall       0.60 1.00   up  +0.40   <- v2's context covers form RF-201
#   winner: v2_grounded

#### 💡 What just happened

⚡ What just happened?Two RAG configurations, one test set, side-by-side scores with deltas - v2 wins on relevance and recall. This is the whole discipline: baseline → change → re-evaluate → deploy if better, rollback if worse. The tradeoff to respect: the judge is noisy, so treat small deltas as ties and change one variable at a time - the ablation matrix in step 8 makes that rigorous. Every RAG improvement from here is backed by a number, not a vibe.

- Set the four metric scores for two RAG configs and watch the winner and the deltas update.
- See how a small, noisy delta reads as a tie, not a win.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## The 2026 Evaluation Stack

You now own the metrics. Steps 5–10 map them onto the tools and disciplines you ship with: DeepEval’s pytest gates, rank-aware retrieval metrics, production observability, component isolation and ablation, LLM-judge biases and human calibration, and India-specific multilingual/legal eval.

---

## 🎯 Concept 5: DeepEval - Evaluation as pytest

### DeepEval - Evaluation as pytest

Gate CI on RAG quality. GEval custom metrics. The referenceless RAG triad.

**Why this is step 5:** RAGAS is great for experiments, but production teams want evaluation in the SAME place as their other tests - the CI pipeline. DeepEval treats every LLM output as an assertion you can gate a build on: write a pytest test, add metrics with thresholds, and the build fails if faithfulness drops below your bar. Plus `GEval` lets you invent any custom metric from a plain-English criterion.

> ✅ **Analogy**
>
> **DeepEval is a unit test for answers.** You already block a merge when a function test fails; DeepEval blocks a merge when an ANSWER test fails - “faithfulness must be ≥ 0.8” is just another assertion. Same pytest muscle memory, same red-build-blocks-deploy safety, now pointed at RAG quality.
> **Where the analogy breaks down:** a unit test is deterministic - same input, same pass/fail forever; an LLM-judged metric is not, so a test can flake red or green on judge randomness. You set thresholds with headroom, pin the judge model, and expect occasional flakes - treat a single red as a signal to look, not an automatic block, unless it repeats.

You want a bad RAG answer to FAIL your CI build, like a failing unit test. Which tool fits?

| Dimension | RAGAS | DeepEval |
|---|---|---|
| Core RAG metrics | Research-backed originals | Re-implemented + a RAGAS wrapper |
| Custom metrics | Limited | GEval from a plain-English criterion |
| Safety (toxicity/bias/PII) | None | Built-in |
| CI/CD | Manual scripts | Native pytest,assert_test |
| Multi-turn | Basic | ConversationalTestCase, per-turn context |

- An `LLMTestCase` holds the input, the actual output, and the retrieval context.

- The referenceless RAG triad (`FaithfulnessMetric`, `AnswerRelevancyMetric`, `ContextualRelevancyMetric`) needs no gold answer; `GEval` adds a custom criterion.

- `assert_test` fails the build if any metric is below its threshold - run with `deepeval test run`.

**📝 `05_deepeval.py`** - *DeepEval*

In [ ]:
# DEEPEVAL - evaluation as pytest; gate the CI build on RAG quality
# pip install deepeval  ;  run with:  deepeval test run test_rag.py
from deepeval import assert_test
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric, ContextualRelevancyMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

def test_refund_answer():
    case = LLMTestCase(
        input="What is the refund policy?",
        actual_output="Refunds are processed within 14 business days via form RF-201.",
        retrieval_context=["Refunds processed within 14 business days. Submit form RF-201."])

    # the referenceless RAG triad (faithfulness + answer + context relevancy) - no gold answer needed
    faithfulness = FaithfulnessMetric(threshold=0.8)
    relevancy = AnswerRelevancyMetric(threshold=0.8)
    context_rel = ContextualRelevancyMetric(threshold=0.7)   # the third triad member: is the retrieved context relevant?
    # a CUSTOM metric from a plain-English criterion
    conciseness = GEval(name="Conciseness", threshold=0.7,
        criteria="Is the answer concise and free of filler?",
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT])

    assert_test(case, [faithfulness, relevancy, context_rel, conciseness])   # red build if any metric < threshold

# Output: (deepeval test run)
# test_rag.py::test_refund_answer PASSED
#   Faithfulness 1.00 | Answer Relevancy 0.94 | Contextual Relevancy 0.90 | Conciseness 0.88   (all pass)

#### 💡 What just happened

⚡ What just happened?The same RAG quality became a pytest test that fails the build if faithfulness, relevancy, or your custom conciseness metric drops below threshold - evaluation living in CI, next to your other tests. The referenceless RAG triad means you don’t even need gold answers for a quick gate, and `GEval` turns any plain-English criterion into a metric. The rule of thumb: use RAGAS for research experiments, DeepEval for CI gates, and both together via **DeepEval**’s RAGAS wrapper ([github.com/confident-ai/deepeval](https://github.com/confident-ai/deepeval)) - but set thresholds with headroom because the judge flakes. The tradeoff: a strict threshold catches more regressions but costs more false-alarm red builds, so tune it to your team’s tolerance.

---

## 🎯 Concept 6: Retrieval Metrics - NDCG, MRR, Precision@k, Recall@k

### Retrieval Metrics - NDCG, MRR, Precision@k, Recall@k

If retrieval fails, no LLM quality can save you. Evaluate the retriever separately, rank-aware.

**Why this is step 6:** RAGAS’s context precision/recall are LLM-judged and coarse; they don’t care about ORDER. But rank matters enormously - a relevant doc at position 1 is worth far more than the same doc at position 8. Rank-aware retrieval metrics (MRR, NDCG) catch ordering problems that RAGAS misses, and they’re cheap, deterministic Python - no LLM judge, no noise.

> 🏆 **Analogy**
>
> **NDCG grades a search results page like a teacher grading with partial credit and a seating chart.** A perfect answer in the top slot earns full marks; the same answer buried at rank 8 earns less (nobody scrolls that far); a mediocre-but-relevant result earns partial credit. NDCG rolls graded relevance AND position into one number - which is why it’s the MTEB gold standard, and why “Precision@k” (which ignores order) can look fine while your best doc sits at the bottom.
> **Where the analogy breaks down:** a teacher knows the true grade of each answer; NDCG needs YOU to supply graded relevance labels, which are expensive and subjective to produce. In practice teams use binary relevance (relevant / not) and MRR/Recall@k, reserving graded NDCG for a small, carefully-labelled benchmark - the metric is only as good as the labels behind it.

Two retrievers return the SAME relevant docs, but retriever B ranks the best one first instead of last. Which metric notices?

- **precision@k / recall@k** count relevant docs in the top-k - noise vs coverage, order-blind.

- **MRR** = 1 / rank of the first relevant doc - rank-aware, great for single-answer QA.

- **NDCG@k** discounts each doc’s graded relevance by `log2(rank+1)` and normalizes by the ideal order - the gold standard.

**📝 `06_retrieval_metrics.py Pure`** - *Python*

In [ ]:
# RETRIEVAL METRICS FROM SCRATCH - evaluate the RETRIEVER, no LLM judge, no noise
import math

def precision_at_k(retrieved, relevant, k):
    return sum(d in relevant for d in retrieved[:k]) / k

def recall_at_k(retrieved, relevant, k):
    return sum(d in relevant for d in retrieved[:k]) / len(relevant)

def mrr(retrieved, relevant):
    for i, d in enumerate(retrieved, 1):
        if d in relevant: return 1.0 / i        # reciprocal rank of the FIRST relevant doc
    return 0.0

def ndcg_at_k(retrieved, gains, k):
    def dcg(order):
        return sum(gains.get(d, 0) / math.log2(i + 1) for i, d in enumerate(order[:k], 1))
    ideal = sorted(gains, key=gains.get, reverse=True)   # best possible ranking
    return dcg(retrieved) / dcg(ideal) if dcg(ideal) else 0.0

retrieved = ["d3", "d1", "d7", "d2"]            # the ranked results
relevant  = {"d1", "d2"}                         # binary relevant set
gains     = {"d1": 3, "d2": 2, "d3": 0, "d7": 1}   # graded relevance

print(f"Precision@2: {precision_at_k(retrieved, relevant, 2):.2f}")
print(f"Recall@4:    {recall_at_k(retrieved, relevant, 4):.2f}")
print(f"MRR:         {mrr(retrieved, relevant):.2f}")
print(f"NDCG@4:      {ndcg_at_k(retrieved, gains, 4):.2f}")

# Output:
# Precision@2: 0.50   (1 of the top-2 is relevant)
# Recall@4:    1.00   (both relevant docs are in the top-4)
# MRR:         0.50   (first relevant doc is at rank 2 -> 1/2)
# NDCG@4:      0.68   (relevant docs are present but poorly ordered)

#### 💡 What just happened

⚡ What just happened?Four retrieval metrics, pure Python, no LLM judge. Precision and recall counted WHICH docs were retrieved; MRR and NDCG cared about WHERE they ranked - NDCG@4 = 0.68 says the right docs are present but ordered badly (the best doc, d1, sits at rank 2 behind an irrelevant one). That’s the ordering problem RAGAS’s coarse context precision would miss. Always evaluate retrieval separately from generation: if these numbers are low, no prompt tuning will save your answers - the LLM is being fed the wrong or badly-ranked context.

- Drag relevant docs up and down the ranking and watch NDCG rise or fall while precision stays flat.
- Change a doc’s graded relevance and see the ideal-order normalization update.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Production Observability - Watch Quality After You Ship

### Production Observability - Watch Quality After You Ship

Offline eval gates a deploy; observability catches drift in production. Langfuse, Phoenix, LangSmith, TruLens.

**Why this is step 7:** passing your eval set proves the RAG was good on your test questions the day you shipped. Production is different - real queries drift, documents change, the model updates under you. Observability platforms trace every request, sample faithfulness on live traffic, and alert when quality or cost drifts, so you catch a regression from your dashboard instead of from an angry user.

> 📡 **Analogy**
>
> **Offline eval is the pre-flight checklist; observability is the cockpit instruments.** You run the checklist once before takeoff (the eval gate), but you watch the altimeter and fuel gauge the whole flight (traces, live faithfulness, cost). A plane that passed pre-flight can still ice up mid-air; a RAG that passed eval can still drift when the corpus or query mix shifts.
> **Where the analogy breaks down:** cockpit instruments read hard physical truth; your production “faithfulness” gauge is still an LLM judge sampled on a slice of traffic, so it lags and estimates. You alert on TRENDS (faithfulness dropped roughly 10% week-over-week, cost spiked ~150%) rather than single readings, and you keep humans reviewing a sample - the dashboard points you at problems, it doesn’t certify correctness.

Your RAG passed its eval set at launch. Two months later users complain. What most likely happened, and would offline eval catch it?

| Platform | License / hosting | Best for |
|---|---|---|
| Langfuse | MIT, self-host (ClickHouse) | Cost-conscious teams; best token-level cost tracking |
| Arize Phoenix | OTEL-native, self-host | Native RAGAS integration; drift detection |
| LangSmith | Commercial | LangChain-native teams (fastest traces, more lock-in) |
| TruLens | Open source | The RAG triad (context relevance / groundedness / answer relevance) |

- You don’t alert on a single reading - you compare today against a rolling average.

- A faithfulness drop or a cost spike relative to the 7-day average fires an alert.

**📝 `07_monitoring.py Pure`** - *Python*

In [ ]:
# PRODUCTION MONITORING - alert on quality/cost TRENDS, not single readings
def should_alert(faith_today, faith_7day_avg, cost_today, cost_7day_avg):
    alerts = []
    if faith_today < faith_7day_avg - 0.10:      # faithfulness drift
        alerts.append("FAITHFULNESS DROP")
    if cost_today > cost_7day_avg * 1.5:         # cost spike
        alerts.append("COST SPIKE")
    return alerts or ["ok"]

# platforms that trace + score live traffic: Langfuse (github.com/langfuse/langfuse,
# MIT, self-host), Arize Phoenix (OTEL-native, native RAGAS), LangSmith, TruLens.
print(should_alert(faith_today=0.79, faith_7day_avg=0.92, cost_today=180, cost_7day_avg=100))

# Output:
# ['FAITHFULNESS DROP', 'COST SPIKE']

#### 💡 What just happened

⚡ What just happened?Observability moves evaluation from “once, offline” to “continuously, in production.” You trace requests, sample faithfulness on a fraction of live traffic, track P95 latency and daily cost, and alert on trends (error rate up, faithfulness down, cost spike). The platform choice is a real tradeoff: Langfuse for cost-conscious self-hosting, Phoenix for OTEL + native RAGAS, LangSmith for LangChain-native convenience, TruLens for the RAG triad - benchmark on your own volume and verify current pricing before committing.

---

## 🎯 Concept 8: Component Isolation & Ablation - Change One Thing, Measure Everything

### Component Isolation & Ablation - Change One Thing, Measure Everything

Separate retrieval from generation so you fix the right half. Ablate one variable at a time.

**Why this is step 8:** the misconception at the top of this lesson - “high faithfulness means good RAG” - is really a plea for component isolation. A single overall score hides which half is broken. If you evaluate retrieval (NDCG/recall) and generation (faithfulness) SEPARATELY, the failure signature is unambiguous: high faithfulness + bad answers means retrieval is feeding the model the wrong context, not that the model is hallucinating.

> 🔧 **Analogy**
>
> **Component isolation is a mechanic testing systems one at a time.** Car won’t start? A good mechanic doesn’t replace the whole engine - they check the battery, then the starter, then the fuel line, isolating the one broken part. You do the same: is retrieval bringing the right docs (check NDCG)? Is generation faithful to them (check faithfulness)? The combination of readings names the culprit.
> **Where the analogy breaks down:** a car’s systems are mostly independent; RAG components INTERACT - an ablation study famously found several improvements gave ~0% alone but a large gain only when combined. So isolation tells you WHERE a component is weak, but you still have to test combinations (and keep a random-retrieval negative control) to know what actually helps end-to-end.

Your RAG gives bad answers, but faithfulness is 0.92. Which component is the problem?

- `diagnose` takes a retrieval score and generation scores and names the failing component.

- High faithfulness + low retrieval → RETRIEVAL is the culprit (the trap the misconception warns about).

- Low faithfulness + good retrieval → GENERATION is hallucinating - a different fix entirely.

**📝 `08_component_eval.py Pure`** - *Python*

In [ ]:
# COMPONENT ISOLATION - the metric combination names the broken half
def diagnose(retrieval_score, faithfulness, answer_relevance):
    issues = []
    if retrieval_score < 0.7:
        issues.append("RETRIEVAL: wrong/missing chunks (fix chunking, top_k, reranking)")
    if faithfulness < 0.7:
        issues.append("GENERATION: hallucinating (stronger grounding prompt, lower temperature)")
    if retrieval_score >= 0.7 and faithfulness >= 0.9 and answer_relevance < 0.7:
        issues.append("PROMPT: grounded but off-topic (improve the answer template)")
    return issues or ["all components healthy"]

# case A: faithfulness HIGH but answers bad -> blame RETRIEVAL, not the LLM
print("case A:", diagnose(retrieval_score=0.4, faithfulness=0.92, answer_relevance=0.6))
# case B: retrieval good, faithfulness low -> the GENERATOR is hallucinating
print("case B:", diagnose(retrieval_score=0.9, faithfulness=0.5, answer_relevance=0.8))

# Output:
# case A: ['RETRIEVAL: wrong/missing chunks (fix chunking, top_k, reranking)']
# case B: ['GENERATION: hallucinating (stronger grounding prompt, lower temperature)']

#### 💡 What just happened

⚡ What just happened?The metric COMBINATION named the broken component: case A (faithful but low retrieval) is a retrieval bug - the model is dutifully summarizing the wrong chunks; case B (good retrieval, low faithfulness) is a generation bug. This is the whole reason you never ship on one overall score. Then ablation makes it rigorous: change ONE variable, hold the rest fixed, record the delta in a matrix, and keep a random-retrieval negative control - because components interact, and the only proof an improvement works is a controlled measurement, not intuition. The tradeoff of that rigor: ablation is slower and costs more eval runs than changing everything at once, but it is the only way to know which change actually helped instead of guessing.

> 💡 **Info**
>
> 💰 The cost of automated eval - a tradeoff you must budget for
> Every metric on this page is an LLM call. Four metrics over a test set of a few hundred questions is well over a thousand judge calls *per pipeline version* - and an A/B run scores two versions, and CI re-runs on every pull request. That is easily thousands of judge calls (and dollars, and minutes) per iteration.
> - **Use a cheap, fast judge for bulk coverage** (a Flash-tier model) and reserve a stronger or second judge only for the human-calibration sample from step 9.
> - **Sample, don’t exhaust:** gate CI on a small curated set (roughly 50 to 100 questions), run the full suite nightly, and monitor production on a traffic sample - not every request.
> - **Cache** judge verdicts keyed on (question, answer, context) so an unchanged row is never re-scored.

- Slide retrieval quality and generation faithfulness independently and watch the diagnosis flip between RETRIEVAL and GENERATION.
- Find the “high faithfulness, low retrieval” trap the misconception warns about.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 9: LLM-Judge Biases & Human Evaluation - Evaluate the Evaluator

### LLM-Judge Biases & Human Evaluation - Evaluate the Evaluator

The judge has biases. Calibrate it against humans with Cohen’s Kappa; route the doubtful cases to experts.

**Why this is step 9:** every metric so far leaned on an LLM judging another LLM - and that judge is biased. It favors longer answers, prefers whatever it sees first, and rates its own outputs higher. Its agreement with humans drops sharply across languages. So you have to evaluate the evaluator: measure judge-vs-human agreement (Cohen’s Kappa), and build a hybrid pipeline that routes low-confidence cases to people.

> ⚖️ **Analogy**
>
> **Trusting one LLM judge is trusting a single referee with known blind spots.** This ref consistently favors the louder team (verbosity bias), the team that kicks off first (position bias), and its own past calls (self-enhancement). You wouldn’t settle a championship on one biased ref - you add instant replay for close calls (human review) and check how often the ref agrees with the rulebook (Cohen’s Kappa).
> **Where the analogy breaks down:** a sports ref’s biases are roughly constant; an LLM judge’s biases shift with language and domain - its human-agreement can fall to near-chance on Hindi or on legal citations. So you can’t just apply one correction; you calibrate PER domain and language, and for high-stakes content (legal, medical) you keep humans in the loop rather than trusting any judge score.

You show an LLM judge the same two answers twice, but swap their order. Should the verdict change?

> 📦 **Info**
>
> 👤 The judge-plus-human pipeline
> - **Known judge biases:** position (order of options shifts the verdict), verbosity (longer looks better), self-enhancement (favors its own style). Documented across multiple studies. **Concrete rule:** never use the same model that GENERATED an answer as the judge that scores it - pick a different model family (or a stronger judge) so self-enhancement bias cannot inflate your numbers.
> - **Measure agreement:** Cohen’s Kappa (2 raters) or Krippendorff’s Alpha (3+) - agreement BEYOND chance. A common bar for “substantial” agreement is around 0.6+.
> - **Hybrid loop:** LLM-judge everything → route low-confidence scores (e.g. faithfulness < 0.7) to human review → spot-check a sample of the passes → feed corrections back to calibrate the judge.
> - **High stakes = humans:** for legal/medical/regulatory answers, citation precision exceeds what any LLM judge can verify - keep experts in the loop.

#### 💡 What just happened

⚡ What just happened?The uncomfortable truth of automated eval: the judge is biased (position, verbosity, self-enhancement) and its agreement with humans is far from perfect, especially across languages. So you calibrate it - measure Cohen’s Kappa against a human-labelled sample - and build a hybrid pipeline that auto-scores everything but routes the doubtful cases to experts and feeds their corrections back. Even a handful of human reviews a week meaningfully improves the whole system. The tradeoff to accept: automated metrics scale but are biased and coarse, while human judgment is accurate but slow and expensive - so you use automated eval for coverage and reserve humans for calibration and the doubtful cases.

- Dial up position and verbosity bias and watch the judge’s verdicts drift away from the human labels.
- See Cohen’s Kappa fall from “substantial” toward “near-chance” as bias grows.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 10: India RAG Evaluation - Hindi, Hinglish, and Legal Precision

### India RAG Evaluation - Hindi, Hinglish, and Legal Precision

Where LLM judges fail hardest: multilingual and high-stakes regulatory content.

**Why this is step 10:** everything hard about LLM-as-judge gets harder in India. Hindi and code-mixed Hinglish tokenize inefficiently and degrade both retrieval and the judge; the judge’s cross-language agreement drops toward chance; and Indian regulatory RAG hinges on citation precision (“Section 17(5)(d)” vs “17(5)(c)”) that changes the legal meaning entirely and that no LLM judge reliably verifies.

> 🇮🇳 **Analogy**
>
> **An LLM judge on Hindi legal text is a foreign referee officiating a match under unfamiliar rules.** They’ll get the obvious calls right, but they miss the local nuance and the fine print - and in law, the fine print (one sub-clause letter) IS the ruling. You bring in a local expert (a lawyer, a native speaker) for the calls that actually matter.
> **Where the analogy breaks down:** a foreign ref can learn the rules; the LLM judge’s weakness is structural - tokenization and training-data scarcity mean it degrades on Devanagari no matter how you prompt it. So the fix isn’t “a better prompt” but a different pipeline: transliterate and use Indic-tuned embeddings for retrieval, benchmark on Indian sets (MIRAGE-Bench, IL-TUR), and route legal answers to human experts - automated eval screens, humans decide.

A Hindi legal answer cites “Section 17(5)(c)” but the correct sub-clause is “17(5)(d)”. Will an LLM-judge faithfulness score catch it?

> 📦 **Info**
>
> 🌐 India evaluation patterns
> - **Language:** Hindi/Hinglish tokenize inefficiently and are under-represented in judge training data, so LLM-judge agreement drops sharply. Transliterate Romanized Hindi to Devanagari and use Indic-tuned embeddings before you even measure.
> - **Legal precision:** a single sub-clause letter (“17(5)(d)” vs “17(5)(c)”) changes the meaning - exact-citation checks and human expert review, not an LLM judge, for regulatory answers.
> - **Benchmarks:** MIRAGE-Bench (multilingual RAG including Indian languages) and IL-TUR (Indian legal NLP tasks) probe exactly the failure modes generic English benchmarks miss - benchmark on Indian sets, not just MTEB.
> - **Pipeline:** automated screening (RAGAS/DeepEval) for bulk → threshold-route low scores and all legal citations to expert review → calibrate the judge with expert feedback. Human-in-the-loop is not optional here.

#### 💡 What just happened

⚡ What just happened?India evaluation is where automated eval’s limits bite hardest: the LLM judge is weakest exactly where the stakes are highest - a different language and legal text where one sub-clause letter is the whole answer. The pattern is the hybrid pipeline from step 9, tilted toward humans: transliterate and use Indic embeddings for retrieval, benchmark on Indian sets (MIRAGE-Bench, IL-TUR), auto-screen the bulk, and route every legal citation to a human expert. Automated metrics get you coverage; for high-stakes multilingual content, human judgment is the ground truth.

**📝 `10_citation_check.py Pure`** - *Python*

In [ ]:
# INDIA LEGAL EVAL - exact citation checking (a fluency-based judge misses this)
import re
CITATION = re.compile(r"Section\s+\d+\(\d+\)\([a-z]\)")

def citations_match(answer, reference):
    a, r = set(CITATION.findall(answer)), set(CITATION.findall(reference))
    return a == r, (r - a)      # exact match? and which references are missing/wrong

ok, missing = citations_match(
    answer="Input tax credit is blocked under Section 17(5)(c).",
    reference="Input tax credit is blocked under Section 17(5)(d).")   # ONE letter differs
print("citations match:", ok, "| missing/wrong:", missing)

# A fluent, grounded-sounding answer with the WRONG sub-clause is a legal error a judge rates as fine.
# The tradeoff: exact checks are strict and language-specific - when to use them is exactly here,
# high-stakes regulatory text, alongside a human expert.

# Output:
# citations match: False | missing/wrong: {'Section 17(5)(d)'}

## Putting It Together

You turned “it seems to work” into engineering: the four core metrics by hand, the RAGAS and DeepEval frameworks, synthetic test sets, rank-aware retrieval metrics, production observability, component isolation, and human calibration. The through-line: *evaluation is how you know - every RAG decision should be backed by a number, every number should be checked against a second one (retrieval vs generation, judge vs human), and no single score is ever the whole truth.*

> 📦 **Info**
>
> 🔑 Key takeaways
> - **Measure, don’t eyeball.** The four core metrics (faithfulness, answer relevance, context precision, context recall) turn hunches into numbers you can gate on.
> - **Faithfulness measures grounding, not truth.** High faithfulness + bad answers = a retrieval failure; evaluate retrieval and generation SEPARATELY.
> - **Use the current APIs.** RAGAS v0.2+ (`EvaluationDataset`, metric classes, renamed fields) for experiments; DeepEval’s pytest gates for CI.
> - **Rank matters.** RAGAS’s context metrics are order-blind; add NDCG/MRR to catch ranking problems - cheap, deterministic, no judge noise.
> - **A/B with discipline.** Change one variable, re-evaluate on the same set, treat small deltas as ties, keep a negative control.
> - **Evaluate the evaluator.** The LLM judge is biased and weakest on other languages and high-stakes content - calibrate with Cohen’s Kappa and keep humans in the loop.

> ✅ **Info**
>
> 🗺️ Where this goes next
> - In Module 9 (**Fine-Tuning**) these same metrics become the before/after scorecard that proves a fine-tune actually helped - you never fine-tune without an eval baseline.
> - In Module 14 (**LLMOps**) offline eval becomes a CI gate and production monitoring - the observability of step 7 wired into dashboards, alerts, and error-analysis loops (Module 14.4).
> - In Module 13 (**Cost & Performance**) you trade eval scores against latency and cost - the metric that’s “good enough” at a fraction of the price.

*Practice exercises are in the companion practice notebook.*

Eight exercises, easy to hard. Each builds or operates one layer of the evaluation stack.

---

## 🎓 Summary

You've completed the practical part of **RAG Evaluationwith RAGAS**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-8_11.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-8.11.html` - regenerate this notebook whenever the source HTML is updated.*
